# How far off was the bike-rental forecast?

Before the morning rush, a bike-share operator could use a forecast of how many rentals will happen in the coming hour to plan staffing. After the hour, the rental log tells us how many actually happened. This walkthrough fits a simple forecast on actual hourly counts from Capital Bikeshare in Washington, DC and evaluates it on later recorded hours.

**Classroom flow.** Run all cells, then open the results in order. First ask learners to estimate a single hour and reveal its actual count. Use the five worked hours to calculate error metrics. Finish with the full later period, which shows performance beyond one morning.

**Source.** Hadi Fanaee-T (2013), *Bike Sharing*, UCI Machine Learning Repository. The supplied example rows and scores are derived from the official **hour.csv**. [Dataset and attribution](https://doi.org/10.24432/C5W894) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) · [Capital Bikeshare system data](https://capitalbikeshare.com/system-data).

## Fit on earlier hours and keep later hours for evaluation

The model uses the hour and whether the day was a workday. For a new hour, average rental totals from earlier hours in the same group and round to the nearest rental. This model is a straightforward baseline built for class. The records contain weather, casual rentals, and registered rentals. Casual plus registered equals our target, so feeding those two counts into the prediction would give the model the answer. The example uses only calendar information available beforehand.

The dates matter. We train through October 31, 2012. November and December 2012 stay held out. Five consecutive hours on November 1 make the calculations readable, then we score all 1,460 later recorded hours.

In [ ]:
import csv
import io
import json
import math
import zipfile
from collections import defaultdict
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from IPython.display import HTML, display

SOURCE = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
TEST_START = '2012-11-01'
LESSON_HOURS = range(7, 12)

# Embedded output lets the live walkthrough run during an internet interruption.
CACHED = json.loads('{"source":{"citation":"Fanaee-T, H. (2013). Bike Sharing [Dataset]. UCI Machine Learning Repository.","doi":"https://doi.org/10.24432/C5W894","archive":"https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip","license":"https://creativecommons.org/licenses/by/4.0/"},"train":{"start":"2011-01-01","end":"2012-10-31","n":15919},"test":{"start":"2012-11-01","end":"2012-12-31","n":1460},"rows":[{"case_id":"07:00","date":"2012-11-01","hour":7,"workingday":1,"actual":420,"predicted":289,"train_group_count":456,"train_group_total":131628},{"case_id":"08:00","date":"2012-11-01","hour":8,"workingday":1,"actual":680,"predicted":470,"train_group_count":456,"train_group_total":214195},{"case_id":"09:00","date":"2012-11-01","hour":9,"workingday":1,"actual":331,"predicted":238,"train_group_count":456,"train_group_total":108510},{"case_id":"10:00","date":"2012-11-01","hour":10,"workingday":1,"actual":146,"predicted":134,"train_group_count":456,"train_group_total":61078},{"case_id":"11:00","date":"2012-11-01","hour":11,"workingday":1,"actual":170,"predicted":157,"train_group_count":456,"train_group_total":71432}],"baseline":189,"metric_full_test":{"calendar_model":{"mae":63.51575342465753,"mse":9611.37602739726,"rmse":98.03762556996809,"r2":0.6841354141080669},"training_mean":{"mae":140.08424657534246,"mse":30428.876027397262,"rmse":174.43874577454764,"r2":-2.944235400592632e-06}}}')

In [ ]:
def score(actual, forecast):
    assert len(actual) == len(forecast)
    residuals = [y - guess for y, guess in zip(actual, forecast)]
    n = len(actual)
    mean = sum(actual) / n
    sse = sum(e * e for e in residuals)
    sst = math.fsum((y - mean) ** 2 for y in actual)
    return {
        "mae": sum(abs(e) for e in residuals) / n,
        "mse": sse / n,
        "rmse": math.sqrt(sse / n),
        "r2": 1 - sse / sst,
    }


def build(archive):
    with zipfile.ZipFile(io.BytesIO(archive)) as source:
        with source.open("hour.csv") as file:
            records = [
                (row["dteday"], int(row["hr"]), int(row["workingday"]), int(row["cnt"]))
                for row in csv.DictReader(io.TextIOWrapper(file, encoding="utf-8"))
            ]
    records.sort()
    assert len(records) == 17379
    assert len({(date, hour) for date, hour, _, _ in records}) == len(records)
    train = [row for row in records if row[0] < TEST_START]
    test = [row for row in records if row[0] >= TEST_START]

    groups = defaultdict(list)
    for _, hour, workingday, actual in train:
        groups[hour, workingday].append(actual)

    def predict(hour, workingday):
        group = groups[hour, workingday]
        return round(sum(group) / len(group))

    rows = []
    for date, hour, workingday, actual in test:
        if date == TEST_START and hour in LESSON_HOURS:
            group = groups[hour, workingday]
            rows.append({
                "case_id": f"{hour:02d}:00",
                "date": date,
                "hour": hour,
                "workingday": workingday,
                "actual": actual,
                "predicted": predict(hour, workingday),
                "train_group_count": len(group),
                "train_group_total": sum(group),
            })
    assert [row["hour"] for row in rows] == list(LESSON_HOURS)

    baseline = round(sum(actual for _, _, _, actual in train) / len(train))
    return {
        "source": {
            "citation": "Fanaee-T, H. (2013). Bike Sharing [Dataset]. UCI Machine Learning Repository.",
            "doi": "https://doi.org/10.24432/C5W894",
            "archive": SOURCE,
            "license": "https://creativecommons.org/licenses/by/4.0/",
        },
        "train": {"start": train[0][0], "end": train[-1][0], "n": len(train)},
        "test": {"start": test[0][0], "end": test[-1][0], "n": len(test)},
        "rows": rows,
        "baseline": baseline,
        "metric_full_test": {
            "calendar_model": score(
                [row[3] for row in test],
                [predict(hour, workingday) for _, hour, workingday, _ in test],
            ),
            "training_mean": score([row[3] for row in test], [baseline] * len(test)),
        },
    }


local_archive = Path('data/bike-sharing-dataset.zip')
try:
    if local_archive.is_file():
        archive = local_archive.read_bytes()
    else:
        with urlopen(SOURCE, timeout=15) as response:
            archive = response.read()
except (OSError, URLError):
    lesson = CACHED
    print('Official archive unavailable. The following visuals use verified saved results.')
else:
    lesson = build(archive)
    assert lesson['rows'] == CACHED['rows']
    assert all(lesson[key] == CACHED[key] for key in ('train', 'test', 'baseline'))
    for name, scores in lesson['metric_full_test'].items():
        assert all(math.isclose(value, CACHED['metric_full_test'][name][key], rel_tol=1e-12, abs_tol=1e-12)
                   for key, value in scores.items())
    print('Rebuilt and verified forecasts from the official UCI archive.')

train, test = lesson['train'], lesson['test']
print(f"Training: {train['n']:,} hours, {train['start']} to {train['end']}")
print(f"Test: {test['n']:,} hours, {test['start']} to {test['end']}")

## Follow the forecast to the observed count

Pick a forecast for November 1. A workday means neither weekend nor public holiday. The colored figure is the forecast before the hour starts. Open the row to show the actual count. A positive residual means that more rentals happened than the model expected. A negative residual means the forecast was too high.

$$
r_i = y_i - \hat{y}_i.
$$

In [ ]:
CSS = '''
<style>
.bike-card { max-width: 780px; border: 1px solid #bad4ca; border-radius: 9px; padding: 10px 14px; margin: 8px 0; color: #193330; background: #f4fbf7; }
.bike-card summary { font-weight: 650; cursor: pointer; color: #0d544c; }
.bike-card p { margin: 8px 0; }
.bike-bar { width: 100%; height: 13px; background: #deece6; border-radius: 4px; margin: 5px 0; }
.bike-bar span { display: block; border-radius: 4px; background: #087669; height: 100%; }
.bike-table { border-collapse: collapse; margin-block: 12px; }
.bike-table th, .bike-table td { padding: 5px 11px; border-bottom: 1px solid #bad4ca; text-align: right; }
.bike-table th:first-child { text-align: left; }
</style>
'''
rows = lesson['rows']
actual = [row['actual'] for row in rows]
forecast = [row['predicted'] for row in rows]
scale = max(actual + forecast)
cards = []
for row in rows:
    label = row['case_id']
    total, count = row['train_group_total'], row['train_group_count']
    predicted, observed = row['predicted'], row['actual']
    residual = observed - predicted
    cards.append(f'''<details class="bike-card">
      <summary>{label} · Forecast {predicted} rentals · Reveal actual count</summary>
      <p>Historical calculation: {total:,} ÷ {count:,} earlier workday hours at {label} ≈ {predicted} rentals after rounding.</p>
      <div class="bike-bar" aria-hidden="true"><span style="width:{100 * predicted / scale}%"></span></div>
      <p>Actual count: <strong>{observed} rentals</strong>.</p>
      <div class="bike-bar" aria-hidden="true"><span style="width:{100 * observed / scale}%"></span></div>
      <p>Residual = {observed} − {predicted} = {residual:+} rentals.</p>
    </details>''')
display(HTML(CSS + ''.join(cards)))

## Score the five displayed forecasts

All tuples pair the forecast and actual for the same hour. Sum signed residuals to see direction, sum absolute residuals to measure typical size, or square the residuals to give larger misses more weight.

- **MAE** $=\frac{\sum_i |y_i-\hat y_i|}{n}$ in rentals.
- **MSE** $=\frac{\sum_i (y_i-\hat y_i)^2}{n}$ in rentals².
- **RMSE** $=\sqrt{\mathrm{MSE}}$ in rentals.

Change a forecast in the cell, rerun it, and explain which metric moves. Restore the original forecast afterward.

In [ ]:
residuals = [y - guess for y, guess in zip(actual, forecast)]
absolute = [abs(error) for error in residuals]
squares = [error ** 2 for error in residuals]
n = len(actual)

mae = sum(absolute) / n
mse = sum(squares) / n
rmse = math.sqrt(mse)
print(f'First residual = {actual[0]} − {forecast[0]} = {residuals[0]:+} rentals')
print(f"MAE = ({' + '.join(map(str, absolute))}) ÷ {n} = {mae:.2f} rentals")
print(f"MSE = ({' + '.join(map(str, squares))}) ÷ {n} = {mse:.2f} rentals²")
print(f'RMSE = √{mse:.2f} ≈ {rmse:.2f} rentals')

## Which metric reacts more to one big miss?

Keep the exact same observed rental counts. Create two controlled forecast sets. One has a 10-rental miss each hour, and one has a 50-rental miss in the last hour only. Ask the class what happens to MAE and RMSE, then reveal the results.

In [ ]:
distributed = [y + 10 for y in actual]
concentrated = actual[:-1] + [actual[-1] + 50]
scenarios = [('Five 10-rental misses', distributed), ('One 50-rental miss', concentrated)]
parts = []
for name, guesses in scenarios:
    errors = [abs(y - guess) for y, guess in zip(actual, guesses)]
    squared = [error ** 2 for error in errors]
    values = score(actual, guesses)
    parts.append(f'''<details class="bike-card">
      <summary>{name} · Reveal the calculation</summary>
      <p>Absolute errors {errors}. MAE = {sum(errors):,} ÷ {n} = {values['mae']:g} rentals.</p>
      <p>Squared errors {squared}. MSE = {sum(squared):,} ÷ {n} = {values['mse']:g} rentals².</p>
      <p>RMSE = √{values['mse']:g} ≈ {values['rmse']:.2f} rentals.</p>
    </details>''')
display(HTML(CSS + ''.join(parts)))

## How much squared error did we save?

For these five hours, the mean of the observed counts is the best constant on the very sample we score. Compare a model's squared misses against this reference.

$$
\mathrm{SSE}=\sum_i(y_i-\hat y_i)^2,\qquad
\mathrm{SST}=\sum_i(y_i-\bar y)^2,\qquad
R^2=1-\frac{\mathrm{SSE}}{\mathrm{SST}}.
$$

The evaluation mean uses observations from after the forecast. A deployable constant forecast instead uses the earlier training mean. If all evaluation targets are equal, SST is zero and the mathematical R² formula has no finite value. Check how your software reports this edge case.

In [ ]:
mean_actual = sum(actual) / n
reference_squares = [(y - mean_actual) ** 2 for y in actual]
sst = sum(reference_squares)
comparisons = [
    ('Calendar model', forecast), ('Evaluation mean', [mean_actual] * n),
    ('Training mean', [lesson['baseline']] * n), ('Always predict zero', [0] * n),
    ('Perfect forecast', actual),
]

parts = [f'<p>Mean actual = ({" + ".join(map(str, actual))}) ÷ {n} = {mean_actual:g} rentals.</p>',
         f'<p>SST = {" + ".join(f"{term:,.2f}" for term in reference_squares)} = {sst:,.2f} rentals².</p>']
for name, guesses in comparisons:
    squares = [(y - guess) ** 2 for y, guess in zip(actual, guesses)]
    sse = sum(squares)
    r2 = 1 - sse / sst
    parts.append(f'''<details class="bike-card">
      <summary>{name} · Reveal R²</summary>
      <p>SSE = {' + '.join(f'{term:,.2f}' for term in squares)} = {sse:,.2f} rentals².</p>
      <p>R² = 1 − {sse:,.2f} ÷ {sst:,.2f} = <strong>{r2:.3f}</strong>.</p>
    </details>''')
display(HTML(CSS + ''.join(parts)))

## Check the full held-out period

The first morning deliberately makes each term visible. We compare predictions across all recorded hours in the held-out period to describe this forecast's broader behavior. MAE gives a typical absolute miss, RMSE is more sensitive to large errors, and R² gives proportional squared-error reduction relative to the mean of the evaluation targets.

The forecast predicts recorded rentals across the system. Those records measure trips that occurred and do not measure every attempted rental or demand at an individual station.

In [ ]:
scores = lesson['metric_full_test']
parts = ['<tr><th>Forecast</th><th>MAE (rentals)</th><th>MSE (rentals²)</th><th>RMSE (rentals)</th><th>R²</th></tr>']
for label, key in [('Calendar model', 'calendar_model'), ('Training mean', 'training_mean')]:
    result = scores[key]
    parts.append(f'''<tr><th scope="row">{label}</th><td>{result['mae']:.2f}</td>
      <td>{result['mse']:,.2f}</td><td>{result['rmse']:.2f}</td><td>{result['r2']:.6f}</td></tr>''')
display(HTML(CSS + '<table class="bike-table">' + ''.join(parts) + '</table>'))